## Supply Chain & Logistics Analytics

#### Load and inspect the data using Pandas

In [1]:
import pandas as pd

df = pd.read_csv(
    "data/DataCoSupplyChainDataset.csv",
    encoding="latin1",
    low_memory=False
)

df.shape

(180519, 53)

In [2]:
# Check the column names

df.columns

Index(['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)',
       'Benefit per order', 'Sales per customer', 'Delivery Status',
       'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City',
       'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id',
       'Customer Lname', 'Customer Password', 'Customer Segment',
       'Customer State', 'Customer Street', 'Customer Zipcode',
       'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market',
       'Order City', 'Order Country', 'Order Customer Id',
       'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id',
       'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id',
       'Order Item Product Price', 'Order Item Profit Ratio',
       'Order Item Quantity', 'Sales', 'Order Item Total',
       'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status',
       'Order Zipcode', 'Product Card Id', 'Product Category Id',
       'Product De

In [3]:
# Inspect the first five records

df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [4]:
# Check missing values

df.isnull().sum().sort_values(ascending=False).head(15)

Product Description              180519
Order Zipcode                    155679
Customer Lname                        8
Customer Zipcode                      3
Days for shipment (scheduled)         0
Sales per customer                    0
Benefit per order                     0
Delivery Status                       0
Late_delivery_risk                    0
Customer City                         0
Customer Country                      0
Category Id                           0
Category Name                         0
Customer Fname                        0
Customer Email                        0
dtype: int64

In [5]:
# Check the shipping modes

df["Shipping Mode"].value_counts(dropna=False)

Shipping Mode
Standard Class    107752
Second Class       35216
First Class        27814
Same Day            9737
Name: count, dtype: int64

In [6]:
# Check the markets

df["Market"].value_counts(dropna=False)

Market
LATAM           51594
Europe          50252
Pacific Asia    41260
USCA            25799
Africa          11614
Name: count, dtype: int64

In [7]:
# Check the shipping departments

df["Department Name"].value_counts(dropna=False)

Department Name
Fan Shop              66861
Apparel               48998
Golf                  33220
Footwear              14525
Outdoors               9686
Fitness                2479
Discs Shop             2026
Technology             1465
Pet Shop                492
Book Shop               405
Health and Beauty       362
Name: count, dtype: int64

#### Prepare the main analysis columns

In [8]:
df["late_delivery"] = (
    df["Days for shipping (real)"]
    > df["Days for shipment (scheduled)"]
).astype(int)

df["late_delivery"]

0         0
1         1
2         0
3         0
4         0
         ..
180514    0
180515    1
180516    1
180517    0
180518    0
Name: late_delivery, Length: 180519, dtype: int64

In [9]:
# Create an order-level table

orders = df.drop_duplicates("Order Id").copy()

In [10]:
# Check how many order records remain

orders["Order Id"].nunique()

65752

#### Business Question 1: Which shipping modes and regions experience the most late deliveries?

In [11]:
# Late delivery by shipping mode

shipping_analysis = (
    orders
    .groupby("Shipping Mode")
    .agg(
        total_orders=("Order Id", "nunique"),
        late_orders=("late_delivery", "sum"),
        late_delivery_rate=("late_delivery", "mean")
    )
    .reset_index()
)

shipping_analysis["late_delivery_rate_pct"] = (
    shipping_analysis["late_delivery_rate"] * 100
)

shipping_analysis.sort_values(
    "late_delivery_rate_pct",
    ascending=False
)

,Shipping Mode,total_orders,late_orders,late_delivery_rate,late_delivery_rate_pct
0,First Class,10079,10079,1.000000,100.000000
2,Second Class,12778,10213,0.799264,79.926436
1,Same Day,3571,1727,0.483618,48.361803
3,Standard Class,39324,15679,0.398713,39.871325


In [12]:
# Late delivery by region

region_analysis = (
    orders
    .groupby("Order Region")
    .agg(
        total_orders=("Order Id", "nunique"),
        late_orders=("late_delivery", "sum"),
        late_delivery_rate=("late_delivery", "mean")
    )
    .reset_index()
)

region_analysis["late_delivery_rate_pct"] = (
    region_analysis["late_delivery_rate"] * 100
)

region_analysis.sort_values(
    "late_delivery_rate_pct",
    ascending=False
)

,Order Region,total_orders,late_orders,late_delivery_rate,late_delivery_rate_pct
2,Central Africa,556,337,0.606115,60.611511
5,East Africa,613,360,0.587276,58.727569
14,South of USA,1345,787,0.585130,58.513011
22,Western Europe,10010,5842,0.583616,58.361638
20,West Asia,2022,1177,0.582097,58.209693
13,South Asia,3335,1939,0.581409,58.140930
15,Southeast Asia,4356,2530,0.580808,58.080808
8,Eastern Europe,1292,748,0.578947,57.894737
18,US Center,1935,1115,0.576227,57.622739
6,East of USA,2323,1334,0.574257,57.425743


#### Business Question 2: Which product categories generate high sales but low profit?

In [13]:
# Identify category profit

category_profit = (
    df
    .groupby("Category Name")
    .agg(
        total_sales=("Sales", "sum"),
        total_profit=("Order Profit Per Order", "sum"),
        total_items=("Order Item Quantity", "sum")
    )
    .reset_index()
)

category_profit["profit_margin_pct"] = (
    category_profit["total_profit"]
    / category_profit["total_sales"] * 100
)

category_profit.sort_values(
    "total_sales",
    ascending=False
)

,Category Name,total_sales,total_profit,total_items,profit_margin_pct
18,Fishing,6.929654e+06,756220.767190,17325,10.912822
12,Cleats,4.431943e+06,494636.919791,73734,11.160724
9,Camping & Hiking,4.118426e+06,427455.568106,13729,10.379101
10,Cardio Equipment,3.694843e+06,383011.098485,37587,10.366099
47,Women's Apparel,3.147800e+06,350421.029567,62956,11.132252
46,Water Sports,3.113845e+06,325146.960038,15540,10.441977
34,Men's Footwear,2.891758e+06,311902.820214,22246,10.785925
30,Indoor/Outdoor Games,2.888994e+06,318451.430554,57803,11.022918
38,Shop By Sport,1.309522e+06,129813.960315,32726,9.913079
13,Computers,6.630000e+05,69656.810171,442,10.506306


In [14]:
# Identify categories with high sales and low profit

high_sales_low_profit = category_profit[
    (category_profit["total_sales"] >= category_profit["total_sales"].median())
    & (category_profit["total_profit"] < category_profit["total_profit"].median())
]

high_sales_low_profit.sort_values(
    "total_sales",
    ascending=False
)

,Category Name,total_sales,total_profit,total_items,profit_margin_pct
31,Kids' Golf Clubs,98797.579234,9045.839896,645,9.155933


#### Business Question 3: How do shipping delays relate to order profitability and customer experience?

In [15]:
# Profitability by delivery status

order_profit = (
    df
    .groupby("Order Id")
    .agg(
        late_delivery=("late_delivery", "max"),
        order_sales=("Sales", "sum"),
        order_profit=("Order Profit Per Order", "sum")
    )
    .reset_index()
)

profit_by_delivery = (
    order_profit
    .groupby("late_delivery")
    .agg(
        total_orders=("Order Id", "nunique"),
        average_order_sales=("order_sales", "mean"),
        average_order_profit=("order_profit", "mean")
    )
    .reset_index()
)

profit_by_delivery

,late_delivery,total_orders,average_order_sales,average_order_profit
0,0,28054,561.757676,61.842743
1,1,37698,557.726807,59.206500


In [16]:
# Customer experience

order_profit = (
    df
    .groupby("Order Id")
    .agg(
        late_delivery=("late_delivery", "max"),
        order_sales=("Sales", "sum"),
        order_profit=("Order Profit Per Order", "mean")
    )
    .reset_index()
)

profit_by_delivery = (
    order_profit
    .groupby("late_delivery")
    .agg(
        total_orders=("Order Id", "nunique"),
        average_order_sales=("order_sales", "mean"),
        average_order_profit=("order_profit", "mean")
    )
    .reset_index()
)

profit_by_delivery

,late_delivery,total_orders,average_order_sales,average_order_profit
0,0,28054,561.757676,23.364378
1,1,37698,557.726807,22.950340


#### Business Question 4: Which markets or departments have recurring fulfilment issues?

In [17]:
# By market

market_analysis = (
    orders
    .groupby("Market")
    .agg(
        total_orders=("Order Id", "nunique"),
        late_orders=("late_delivery", "sum"),
        late_delivery_rate=("late_delivery", "mean")
    )
    .reset_index()
)

market_analysis["late_delivery_rate_pct"] = (
    market_analysis["late_delivery_rate"] * 100
)

market_analysis.sort_values(
    "late_delivery_rate_pct",
    ascending=False
)

,Market,total_orders,late_orders,late_delivery_rate,late_delivery_rate_pct
3,Pacific Asia,17577,10132,0.576435,57.643511
1,Europe,18561,10661,0.574376,57.437638
4,USCA,8579,4919,0.573377,57.337685
2,LATAM,17181,9804,0.570630,57.063035
0,Africa,3854,2182,0.566165,56.616502


In [18]:
# By department

department_analysis = (
    df
    .drop_duplicates(["Order Id", "Department Name"])
    .groupby("Department Name")
    .agg(
        total_orders=("Order Id", "nunique"),
        late_delivery_rate=("late_delivery", "mean")
    )
    .reset_index()
)

department_analysis["late_delivery_rate_pct"] = (
    department_analysis["late_delivery_rate"] * 100
)

department_analysis.sort_values(
    "late_delivery_rate_pct",
    ascending=False
)

,Department Name,total_orders,late_delivery_rate,late_delivery_rate_pct
9,Pet Shop,492,0.613821,61.382114
7,Health and Beauty,362,0.585635,58.563536
10,Technology,1465,0.583618,58.361775
4,Fitness,2437,0.580222,58.022158
8,Outdoors,9066,0.575998,57.599824
1,Book Shop,405,0.575309,57.530864
5,Footwear,13009,0.573987,57.398724
3,Fan Shop,41303,0.572646,57.264605
0,Apparel,35190,0.572123,57.212276
6,Golf,25889,0.572096,57.209626


#### Business Question 5: Where should management investigate logistics bottlenecks first?

In [19]:
# Find high-priority market and shipping-mode combinations

bottleneck_analysis = (
    orders
    .groupby(["Market", "Shipping Mode"])
    .agg(
        total_orders=("Order Id", "nunique"),
        late_orders=("late_delivery", "sum"),
        late_delivery_rate=("late_delivery", "mean")
    )
    .reset_index()
)

bottleneck_analysis["late_delivery_rate_pct"] = (
    bottleneck_analysis["late_delivery_rate"] * 100
)

bottleneck_analysis = bottleneck_analysis[
    bottleneck_analysis["total_orders"] >= 100
]

bottleneck_analysis.sort_values(
    ["late_delivery_rate_pct", "late_orders"],
    ascending=False
)

,Market,Shipping Mode,total_orders,late_orders,late_delivery_rate,late_delivery_rate_pct
4,Europe,First Class,2878,2878,1.000000,100.000000
12,Pacific Asia,First Class,2668,2668,1.000000,100.000000
8,LATAM,First Class,2638,2638,1.000000,100.000000
16,USCA,First Class,1338,1338,1.000000,100.000000
0,Africa,First Class,557,557,1.000000,100.000000
14,Pacific Asia,Second Class,3469,2789,0.803978,80.397809
2,Africa,Second Class,715,573,0.801399,80.139860
6,Europe,Second Class,3609,2886,0.799667,79.966750
18,USCA,Second Class,1678,1341,0.799166,79.916567
10,LATAM,Second Class,3307,2624,0.793468,79.346840


In [20]:
# Drop Product Image Column

df = df.drop(columns=["Product Image"])

In [22]:
# Export Dataset without Product Image Column

df.to_csv(
    "data/DataCoSupplyChainDataset_clean.csv",
    index=False,
    encoding="utf-8"
)